# Error Analysis — Video Analytics

Run after a pipeline run (`data/run/tracks.parquet`, `data/run/analytics/summary.json`). The goal is to understand **where detection/tracking/counting fail and why**.

1. Load tracks + summary
2. Track-length distribution (fragmentation)
3. Likely ID switches (large position jumps within a track)
4. Low-confidence / rare-class detections
5. Counting error vs ground truth + failure taxonomy

In [ ]:
import json
import numpy as np
import pandas as pd

tracks = pd.read_parquet('data/run/tracks.parquet')
summary = json.load(open('data/run/analytics/summary.json'))
tracks['cx'] = (tracks.x1 + tracks.x2) / 2
tracks['cy'] = (tracks.y1 + tracks.y2) / 2
summary

In [ ]:
# Track-length distribution — many very short tracks => fragmentation / unstable IDs
lengths = tracks.groupby('track_id').size()
print('tracks:', len(lengths), ' short (<5 frames):', int((lengths < 5).sum()))
lengths.plot(kind='hist', bins=30, title='Track length (frames)')

In [ ]:
# Likely ID switches: unusually large centroid jump between consecutive frames of a track
jumps = []
for tid, g in tracks.sort_values('frame').groupby('track_id'):
    c = g[['cx', 'cy']].to_numpy()
    if len(c) > 1:
        step = np.linalg.norm(np.diff(c, axis=0), axis=1)
        jumps.append({'track_id': tid, 'max_jump_px': float(step.max()), 'frames': len(c)})
jdf = pd.DataFrame(jumps).sort_values('max_jump_px', ascending=False)
jdf.head(15)

In [ ]:
# Confidence + per-class detection volume — where the detector is weak
print(tracks.groupby('class_name')['score'].agg(['count', 'mean']).round(3))
tracks['score'].plot(kind='hist', bins=30, title='Detection confidence (tracked boxes)')

## Counting error + failure taxonomy

Compare `summary['line_counts']` to your manual ground truth, then tag the failures:

| Category | Where | Count | Fix tried | Result |
|---|---|---|---|---|
| Missed small/occluded objects | | | larger model / higher imgsz | |
| ID switch at crossing/overlap | | | ByteTrack/BoT-SORT, higher max_age | |
| Double count (fragmented track) | | | raise min_hits / dedup near line | |
| Line/zone misplacement | | | adjust configs lines/zones | |

**Conclusion (fill in):** which change improved counting accuracy / MOTA the most, and why.